# Media Analytics – ETL Pipeline
### Traditional vs. Digital Media Consumption Shift
#### Business Questions:
1. How have viewership and subscriber metrics trended for traditional vs. digital platforms from 2015–2024?
2. When did the shift from traditional to digital media accelerate most significantly?
3. How does user engagement (hours, sessions, retention) compare across media types and regions?
4. How have streaming subscription prices changed over time, and how do pricing tiers relate to subscriber growth?
5. What factors most influence users to switch from traditional to digital media?

In [7]:
import pandas as pd
import numpy as np
import warnings 
warnings.filterwarnings('ignore')

---
### Part 1 – Data Sources Profile

| # | File | Rows | Feeds |
|---|---|---|---|
| 1 | `streaming_service.csv` | 777 | FACT_SUBSCRIPTION_PRICING, DIM_SUBSCRIPTION_PLAN |
| 2 | `platform_summary.csv` | 12 | DIM_PLATFORM |
| 3 | `platform_financials_comprehensive.csv` | 10 | FACT_MEDIA_PERFORMANCE |
| 4 | `industry_metrics.csv` | 9 | FACT_MEDIA_PERFORMANCE |
| 5 | `traditional_media_viewership_monthly.csv` | 1,624 | FACT_MEDIA_PERFORMANCE |
| 6 | `user_engagement_monthly.csv` | 4,120 | FACT_ENGAGEMENT |
| 7 | `switch_factor_survey.csv` | 2,025 | FACT_ENGAGEMENT, DIM_SWITCH_REASON |
| 8 | `platform_subscriber_monthly.csv` | 640 | FACT_MEDIA_PERFORMANCE |

### 1.1 – streaming_service.csv
**Source:** Kaggle | **Feeds:** FACT_SUBSCRIPTION_PRICING, DIM_SUBSCRIPTION_PLAN  
**Known issues:** Date stored as `Jul-2011` string format — needs parsing

In [14]:
streaming_df = pd.read_csv('data sources/streaming_service.csv')
streaming_df.head(5)

,service,date,price
0,Netflix,Jul-2011,7.99
1,Netflix,Aug-2011,7.99
2,Netflix,Sep-2011,7.99
3,Netflix,Oct-2011,7.99
4,Netflix,Nov-2011,7.99


In [54]:
#shape -- rows/columns
print(f"Number of rows & columns: {streaming_df.shape[0]}, {streaming_df.shape[1]}")

#null count
print(f"\nTotal number of nulls in each column: \n{streaming_df.isnull().sum()}")

#unique services
print(f"\nUnique services within dataframe: {streaming_df['service'].unique()}")

#max/min dates
print(f"\nDate range (raw): {streaming_df['date'].min()} → {streaming_df['date'].max()}")

Number of rows & columns: 777, 3

Total number of nulls in each column: 
service    0
date       0
price      0
dtype: int64

Unique services within dataframe: ['Netflix' 'Disney+' 'HBO Max' 'Hulu' 'Paramount+' 'Prime Video'
 'Apple TV+' 'Peacock' 'Shudder' 'Crunchyroll']

Date range (raw): Apr-2012 → Sep-2023


### 1.2 – platform_summary.csv
**Source:** Streaming Platform Dataset | **Feeds:** DIM_PLATFORM  
**Known issues:** Wide table (36 cols), many NaN — only 5 dim cols needed

In [76]:
platform_df = pd.read_csv('data sources/Streaming Platform Dataset/platform_summary.csv')
platform_df.head(5)

,platform,launch_year,parent_company,content_focus,business_model,reporting_date,subscribers_millions,quarterly_revenue_usd_millions,quarterly_profit_usd_millions,quarterly_eps_usd,...,projected_2026_subscribers_millions,sports_rights_spend_2026_usd_millions,sports_rights_global_share_pct,streaming_revenue_q4_usd_millions,streaming_revenue_growth_pct,monthly_streams_millions,streams_growth_pct,monthly_hours_watched_millions,monthly_hours_growth_pct,peak_concurrent_viewers_millions
0,Netflix,2007,Netflix Inc.,"Original series, films, licensed content",Subscription + Ads,2026-01-19,325.0,12050.0,2410.0,0.56,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Disney+,2019,The Walt Disney Company,"Family, Marvel, Star Wars, National Geographic",Subscription + Ads,2026-02-07,126.0,NaN,NaN,NaN,...,284.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Hulu,2008,The Walt Disney Company,"TV shows, next-day broadcast",Subscription + Ads,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Amazon Prime Video,2006,Amazon.com Inc.,"Original series, movies, sports",Prime bundle + Subscription,2026-02-06,NaN,NaN,NaN,NaN,...,NaN,3800.0,27.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Paramount+,2021,Paramount Global,"Nickelodeon, CBS, Showtime",Subscription + Ads,2026-02-06,NaN,NaN,NaN,NaN,...,NaN,1136.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [78]:
#shape before dim colum extract
print(f"Number of rows & columns (before): {platform_df.shape[0]}, {platform_df.shape[1]}")

#only need 5 dim columns -- for DIM_PLATFORM -- so deriving those
platform_df = platform_df[['platform', 'parent_company', 'business_model', 'content_focus', 'launch_year']]
display(platform_df.head(5))

#shape before dim colum extract
print(f"Number of rows & columns (after): {platform_df.shape[0]}, {platform_df.shape[1]}")

Number of rows & columns (before): 12, 36


,platform,parent_company,business_model,content_focus,launch_year
0,Netflix,Netflix Inc.,Subscription + Ads,"Original series, films, licensed content",2007
1,Disney+,The Walt Disney Company,Subscription + Ads,"Family, Marvel, Star Wars, National Geographic",2019
2,Hulu,The Walt Disney Company,Subscription + Ads,"TV shows, next-day broadcast",2008
3,Amazon Prime Video,Amazon.com Inc.,Prime bundle + Subscription,"Original series, movies, sports",2006
4,Paramount+,Paramount Global,Subscription + Ads,"Nickelodeon, CBS, Showtime",2021


Number of rows & columns (after): 12, 5


In [96]:
#null counts
print(f"\nTotal number of nulls in each column: \n{platform_df.isnull().sum()}")

#unique platforms
print(f"\nUnique platforms within dataframe: {platform_df['platform'].unique()}")
print(f"\nUnique parent companies within dataframe: {platform_df['parent_company'].unique()}")


Total number of nulls in each column: 
platform          0
parent_company    0
business_model    0
content_focus     0
launch_year       0
dtype: int64

Unique platforms within dataframe: ['Netflix' 'Disney+' 'Hulu' 'Amazon Prime Video' 'Paramount+' 'Peacock'
 'Apple TV+' 'Max' 'Roku' 'YouTube TV' 'Tubi' 'Pluto TV']

Unique parent companies within dataframe: ['Netflix Inc.' 'The Walt Disney Company' 'Amazon.com Inc.'
 'Paramount Global' 'Comcast' 'Apple Inc.' 'Warner Bros. Discovery'
 'Roku Inc.' 'Google' 'Fox Corporation']


### 1.3 – platform_financials_comprehensive.csv
**Source:** Streaming Platform Dataset | **Feeds:** FACT_MEDIA_PERFORMANCE  
**Known issues:** 10 rows only (1 per platform, snapshot); many sparse columns

In [102]:
platform_fin_df = pd.read_csv('data sources/Streaming Platform Dataset/platform_financials_comprehensive.csv')
platform_fin_df.head(5)

,platform,reporting_date,subscribers_millions,quarterly_revenue_usd_millions,quarterly_profit_usd_millions,quarterly_eps_usd,annual_content_spend_usd_millions,annual_content_spend_growth_pct,ad_revenue_2025_usd_millions,ad_revenue_2026_projection_usd_millions,...,projected_2026_subscribers_millions,sports_rights_spend_2026_usd_millions,sports_rights_global_share_pct,streaming_revenue_q4_usd_millions,streaming_revenue_growth_pct,monthly_streams_millions,streams_growth_pct,monthly_hours_watched_millions,monthly_hours_growth_pct,peak_concurrent_viewers_millions
0,Netflix,2026-01-19,325.0,12050.0,2410.0,0.56,20000.0,10.0,1500.0,3000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Roku,2026-02-11,98.0,1395.0,80.5,0.53,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Disney+,2026-02-07,126.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,284.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Amazon Prime Video,2026-02-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3800.0,27.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Paramount+,2026-02-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1136.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [110]:
#only taking key financial columns relevant to the fact
platform_fin_df = platform_fin_df[['platform', 'reporting_date', 'subscribers_millions',
                                   'quarterly_revenue_usd_millions', 'quarterly_profit_usd_millions']]
platform_fin_df.head(5)

,platform,reporting_date,subscribers_millions,quarterly_revenue_usd_millions,quarterly_profit_usd_millions
0,Netflix,2026-01-19,325.0,12050.0,2410.0
1,Roku,2026-02-11,98.0,1395.0,80.5
2,Disney+,2026-02-07,126.0,NaN,NaN
3,Amazon Prime Video,2026-02-06,NaN,NaN,NaN
4,Paramount+,2026-02-06,NaN,NaN,NaN


In [137]:
print('Shape:', platform_fin_df.shape)
print('Platforms:', list(platform_fin_df['platform']))
print('\nNull counts (key cols):')
print(platform_fin_df.isnull().sum())

Shape: (10, 5)
Platforms: ['Netflix', 'Roku', 'Disney+', 'Amazon Prime Video', 'Paramount+', 'YouTube TV', 'AMC Networks', 'ITVX', 'DAZN', 'Twitch']

Null counts (key cols):
platform                          0
reporting_date                    0
subscribers_millions              6
quarterly_revenue_usd_millions    7
quarterly_profit_usd_millions     8
dtype: int64


### 1.4 – platform_subscriber_monthly.csv
**Source:** Synthetic (AI-generated, anchored to real quarterly figures) | **Feeds:** FACT_MEDIA_PERFORMANCE  
**Known issues:** Null `revenue_usd_millions` for some platforms; seasonal churn spikes built in

In [140]:
platform_subs_df = pd.read_csv('data sources/platform_subscriber_monthly.csv')

print('Rows & Columns:', platform_subs_df.shape)
print('Columns:', list(platform_subs_df.columns))

print('\nNull counts:')
print(platform_subs_df.isnull().sum())

print('\nSample rows:')
display(platform_subs_df.head(5))

print('\nPlatforms covered:', platform_subs_df['platform_name'].unique())
print('Date range:', platform_subs_df['year_month'].min(), '→', platform_subs_df['year_month'].max())

Rows & Columns: (640, 8)
Columns: ['year_month', 'platform_name', 'subscribers_millions', 'revenue_usd_millions', 'churn_rate_pct', 'new_subscribers_millions', 'cancelled_subscribers_millions', 'country_region']

Null counts:
year_month                          0
platform_name                       0
subscribers_millions                0
revenue_usd_millions              344
churn_rate_pct                      0
new_subscribers_millions            0
cancelled_subscribers_millions      0
country_region                      0
dtype: int64

Sample rows:


,year_month,platform_name,subscribers_millions,revenue_usd_millions,churn_rate_pct,new_subscribers_millions,cancelled_subscribers_millions,country_region
0,2015-01,Netflix,65.32,729.9,0.0596,0.73,0.32,US
1,2015-02,Netflix,64.85,719.2,0.0393,0.74,0.21,US
2,2015-03,Netflix,64.69,701.7,0.0416,0.48,0.22,US
3,2015-04,Netflix,65.16,700.4,0.0343,0.29,0.19,US
4,2015-05,Netflix,64.34,688.3,0.0409,0.42,0.22,US



Platforms covered: ['Netflix' 'Hulu' 'Amazon Prime Video' 'Disney+' 'Max' 'Apple TV+'
 'Peacock' 'Paramount+']
Date range: 2015-01 → 2024-12


### 1.5 – industry_metrics.csv
**Source:** Streaming Platform Dataset | **Feeds:** FACT_MEDIA_PERFORMANCE  
**Known issues:** Annual granularity only; industry-level aggregates, not platform-level

In [143]:
industry_metrics_df = pd.read_csv('data sources/Streaming Platform Dataset/industry_metrics.csv')

print('Rows & Columns:', industry_metrics_df.shape)
print('\nAll rows:')
display(industry_metrics_df.head(5))
print('\nNull counts:')
print(industry_metrics_df.isnull().sum())

Rows & Columns: (9, 8)

All rows:


,metric_category,metric_name,value_usd_billions,unit,year,source,value_pct,value_usd_millions
0,Global Content Spending,Total Global Content Spend 2025,245.0,USD Billions,2025,Ampere Analysis [citation:7],NaN,NaN
1,Global Content Spending,Total Global Content Spend 2026,255.0,USD Billions,2026,Ampere Analysis [citation:7],NaN,NaN
2,Global Content Spending,Streamer Content Spend 2026,101.0,USD Billions,2026,Ampere Analysis [citation:7],NaN,NaN
3,Global Content Spending,Streamer Share of Global Spend,NaN,Percentage,2026,Ampere Analysis [citation:7],40.0,NaN
4,Sports Rights,Global Streaming Sports Rights Spend 2025,13.2,USD Billions,2025,Ampere Analysis [citation:3],NaN,NaN



Null counts:
metric_category       0
metric_name           0
value_usd_billions    4
unit                  0
year                  0
source                0
value_pct             7
value_usd_millions    7
dtype: int64


### 1.6 – traditional_media_viewership_monthly.csv
**Source:** Synthetic (AI-generated) | **Feeds:** FACT_MEDIA_PERFORMANCE  
**Known issues:** Mixed date formats (`Jan-2010`, `2010/01`, `March 2010`), some `metric_value` stored as strings (`"13.6M"`), inconsistent `media_type` casing, 25 intentional duplicate rows

In [148]:
traditional_media_df = pd.read_csv('data sources/traditional_media_viewership_monthly.csv')

print('Rows & Columns:', traditional_media_df.shape)
print('Columns:', list(traditional_media_df.columns))

print('\nmedia_type unique values:', traditional_media_df['media_type'].unique())

print('\nNull counts:')
print(traditional_media_df.isnull().sum())
print('\nSample rows:')
display(traditional_media_df.head(5))

Rows & Columns: (1624, 7)
Columns: ['report_month', 'platform_name', 'media_type', 'metric_name', 'metric_value', 'market', 'source']

media_type unique values: ['Broadcast TV' 'Cable TV' 'cable tv' 'CABLE TV' 'Print' 'print' 'Radio']

Null counts:
report_month     0
platform_name    0
media_type       0
metric_name      0
metric_value     0
market           0
source           0
dtype: int64

Sample rows:


,report_month,platform_name,media_type,metric_name,metric_value,market,source
0,Jan-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.63,US,Nielsen
1,Feb-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.42,US,Nielsen
2,March 2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.6,Global,Nielsen
3,Apr-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.8M,US,Nielsen
4,May-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.28,US,Nielsen


### 1.7 – switch_factor_survey.csv
**Source:** Synthetic (AI-generated) | **Feeds:** FACT_ENGAGEMENT, DIM_SWITCH_REASON  
**Known issues:** Inconsistent `switched_from` values (`Cable` vs `Cable TV` vs `cable tv`), many null `secondary_switch_reason`, duplicate `respondent_id` across survey years (re-contact design), mixed `satisfaction_score` types

In [168]:
survey_df = pd.read_csv('data sources/switch_factor_survey.csv')

print('Shape:', survey_df.shape)
print('Columns:', list(survey_df.columns))

print('\nNull counts:')
print(survey_df.isnull().sum())
print('\nSample rows:')
display(survey_df.head(5))

#respondents per survey year
print('\nTotal Respondents per Survey Year:')
print(survey_df.groupby('survey_year')['respondent_id'].count())

Shape: (2025, 10)
Columns: ['survey_year', 'respondent_id', 'switched_from', 'switched_to', 'primary_switch_reason', 'secondary_switch_reason', 'age_group', 'region', 'household_income_bracket', 'satisfaction_score']

Null counts:
survey_year                    0
respondent_id                  0
switched_from                  0
switched_to                    0
primary_switch_reason          0
secondary_switch_reason     1105
age_group                      0
region                         0
household_income_bracket       0
satisfaction_score           173
dtype: int64

Sample rows:


,survey_year,respondent_id,switched_from,switched_to,primary_switch_reason,secondary_switch_reason,age_group,region,household_income_bracket,satisfaction_score
0,2015,10228,Broadcast TV,YouTube,Convenience,NaN,18-24,Midwest,$75K-$150K,6.89
1,2015,10051,Broadcast TV,Spotify,Content Selection,Recommendation,55+,South,$75K-$150K,8.00
2,2015,11518,newspaper,Amazon Prime Video,Device Availability,Device Availability,35-44,West,>$150K,8.00
3,2015,10563,Cable,Netflix,Price,NaN,18-24,South,<$35K,8.00
4,2015,10501,Cable,Max,Device Availability,NaN,55+,South,>$150K,6.89



Total Respondents per Survey Year:
survey_year
2015    220
2016    200
2017    216
2018    200
2019    207
2020    199
2021    194
2022    185
2023    203
2024    201
Name: respondent_id, dtype: int64


### 1.8 – user_engagement_monthly.csv
**Source:** Synthetic (AI-generated) | **Feeds:** FACT_ENGAGEMENT  
**Known issues:** Mixed date formats (`2015-01` vs `01/2015`), `retention_rate_pct` stored as decimal in some rows and `"78%"` string in others, nulls for new platforms in early months, 40 intentional duplicate rows

In [174]:
user_engagement_df = pd.read_csv('data sources/user_engagement_monthly.csv')

print('Rows & Columns:', user_engagement_df.shape)

print('\nNull counts:')
print(user_engagement_df.isnull().sum())

print('\nSample rows:')
display(user_engagement_df.head(5))

print('\nMixed date formats (sample):', user_engagement_df['year_month'].unique()[:10])
print('\nretention_rate_pct mixed types (sample):')
print(user_engagement_df['retention_rate_pct'].head(20).tolist())

Rows & Columns: (4120, 8)

Null counts:
year_month                         0
platform_name                      0
media_type                         0
avg_hours_per_user_per_month       0
monthly_active_users_millions      0
avg_sessions_per_user_per_month    0
retention_rate_pct                 0
region                             0
dtype: int64

Sample rows:


,year_month,platform_name,media_type,avg_hours_per_user_per_month,monthly_active_users_millions,avg_sessions_per_user_per_month,retention_rate_pct,region
0,01/2015,Netflix,Streaming,32.75,64.72,9.1,0.9165,North America
1,2015-01,Netflix,Streaming,32.75,64.72,9.1,0.9165,Europe
2,2015-01,Netflix,Streaming,32.75,64.72,9.1,0.9165,APAC
3,03/2015,Netflix,Streaming,31.90,65.18,8.8,0.9268,North America
4,2015-03,Netflix,Streaming,31.90,65.18,8.8,92.7%,Europe



Mixed date formats (sample): ['01/2015' '2015-01' '03/2015' '2015-03' '2015-04' '2015-06' '06/2015'
 '2015-07' '2015-08' '09/2015']

retention_rate_pct mixed types (sample):
['0.9165', '0.9165', '0.9165', '0.9268', '92.7%', '0.9268', '0.9069', '90.7%', '0.9069', '0.8953', '89.5%', '0.8953', '0.9039', '0.9039', '0.9039', '0.9142', '0.9142', '0.9142', '0.9025', '90.2%']
